In [1]:
!pip uninstall -y transformers peft accelerate
!pip install transformers==4.38.2
!pip install accelerate==0.27.2
!pip install peft==0.8.2
!pip install sentencepiece datasets evaluate -q

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: peft 0.19.1
Uninstalling peft-0.19.1:
  Successfully uninstalled peft-0.19.1
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.7/130.7 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 96.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 108.0 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.11.0
    Uninstalling huggingface_hub-1.11.0:
      Successfully uninstalled huggingface_hub-1.11.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Success

In [70]:
# =========================================================
# IMPORT
# =========================================================

import pandas as pd
import torch
import shutil
import re

from sklearn.model_selection import train_test_split

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)

In [27]:
# =========================================================
# HAPUS CHECKPOINT LAMA
# =========================================================

shutil.rmtree("./indobart-qg", ignore_errors=True)

In [31]:
# =========================================================
# LOAD DATASET
# =========================================================

df = pd.read_csv("H5_dataset_ML.csv")

In [32]:
# =========================================================
# BERSIHKAN DATA
# =========================================================

df["input"] = df["input"].astype(str).str.strip()
df["target"] = df["target"].astype(str).str.strip()

df = df[
    (df["input"] != "") &
    (df["target"] != "")
]

df = df.drop_duplicates()

df = df.reset_index(drop=True)

In [33]:
## =========================================================
# INFO
# =========================================================

print("=" * 50)
print("JUMLAH DATA")
print("=" * 50)

print(len(df))

print("\nCONTOH:")
print(df.head())

JUMLAH DATA
4674

CONTOH:
                                               input  \
0   generate siapa: Pagi itu Rina bangun lebih awal.   
1   generate kapan: Pagi itu Rina bangun lebih awal.   
2  generate siapa: Rina merapikan tempat tidur di...   
3  generate apa: Rina merapikan tempat tidur di k...   
4  generate dimana: Rina merapikan tempat tidur d...   

                                            target  
0           Siapa yang bangun pagi itu lebih awal?  
1                              Kapan Rina bangun ?  
2  Siapa yang merapikan tempat tidur di kamar nya?  
3              Apa yang Rina rapikan di kamar nya?  
4             Di mana Rina merapikan tempat tidur?  


In [34]:
# =========================================================
# SPLIT DATASET
# =========================================================

train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42
)

In [64]:
test_df.to_csv(
    "H6_test_dataset.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Test dataset berhasil disimpan")

Test dataset berhasil disimpan


In [35]:
# =========================================================
# CONVERT DATASET
# =========================================================

train_dataset = Dataset.from_pandas(train_df)
valid_dataset = Dataset.from_pandas(valid_df)
test_dataset = Dataset.from_pandas(test_df)

In [40]:
# =========================================================
# LOAD MODEL
# =========================================================

model_name = "facebook/bart-base"

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name
)

config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

In [42]:
# =========================================================
# TOKENISASI
# =========================================================

max_input_length = 128
max_target_length = 64


def preprocess_function(examples):

    inputs = examples["input"]

    targets = examples["target"]

    model_inputs = tokenizer(
        inputs,
        max_length=max_input_length,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        text_target=targets,
        max_length=max_target_length,
        truncation=True,
        padding="max_length"
    )

    # ubah padding jadi -100
    labels["input_ids"] = [

        [
            token if token != tokenizer.pad_token_id else -100
            for token in label
        ]

        for label in labels["input_ids"]
    ]

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [43]:
# =========================================================
# TOKENIZE
# =========================================================

tokenized_train = train_dataset.map(
    preprocess_function,
    batched=True
)

tokenized_valid = valid_dataset.map(
    preprocess_function,
    batched=True
)

tokenized_test = test_dataset.map(
    preprocess_function,
    batched=True
)

Map:   0%|          | 0/3739 [00:00<?, ? examples/s]

Map:   0%|          | 0/467 [00:00<?, ? examples/s]

Map:   0%|          | 0/468 [00:00<?, ? examples/s]

In [46]:
# =========================================================
# CEK HASIL TOKENISASI
# =========================================================

print("\nHASIL TOKENISASI:")
print(tokenized_train[0])

print("\nLABEL SAMPLE:")
print(tokenized_train[0]["labels"][:20])


HASIL TOKENISASI:
{'input': 'generate apa: Vina menemukan empat sudut gambar.', 'target': 'Apa yang Vina temukan?', '__index_level_0__': 2715, 'input_ids': [0, 20557, 877, 6256, 102, 35, 468, 1243, 604, 991, 1350, 260, 2841, 11632, 2628, 417, 1182, 34796, 271, 4, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'label

In [ ]:
# =========================================================
# TRAINING ARGUMENT
# =========================================================

training_args = Seq2SeqTrainingArguments(

    output_dir="./indobart-qg",

    evaluation_strategy="epoch",

    save_strategy="epoch",

    learning_rate=3e-5,

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    num_train_epochs=5,

    predict_with_generate=True,

    logging_steps=10,

    fp16=torch.cuda.is_available(),

    load_best_model_at_end=True
)

In [48]:
# =========================================================
# TRAINER
# =========================================================

trainer = Seq2SeqTrainer(

    model=model,

    args=training_args,

    train_dataset=tokenized_train,

    eval_dataset=tokenized_valid,

    tokenizer=tokenizer
)


/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:450: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [49]:
# =========================================================
# TRAIN
# =========================================================

trainer.train()

Epoch,Training Loss,Validation Loss
1,0.347900,0.294410
2,0.247700,0.196732
3,0.198800,0.188246


Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0, 'forced_eos_token_id': 2}
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0, 'forced_eos_token_id': 2}
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file

Epoch,Training Loss,Validation Loss
1,0.347900,0.294410
2,0.247700,0.196732
3,0.198800,0.188246
4,0.106900,0.196451
5,0.097200,0.176279
6,0.093900,0.169761
7,0.048500,0.191801
8,0.032400,0.196210
9,0.022800,0.186059
10,0.015900,0.194598


Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0, 'forced_eos_token_id': 2}
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0, 'forced_eos_token_id': 2}
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file

TrainOutput(global_step=9350, training_loss=0.15757676068872692, metrics={'train_runtime': 1313.3525, 'train_samples_per_second': 28.469, 'train_steps_per_second': 7.119, 'total_flos': 2849755771699200.0, 'train_loss': 0.15757676068872692, 'epoch': 10.0})

In [50]:
# =========================================================
# SAVE MODEL
# =========================================================

trainer.save_model("model_indobart_qg")

tokenizer.save_pretrained(
    "model_indobart_qg"
)

print("\nMODEL BERHASIL DISIMPAN")

Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0, 'forced_eos_token_id': 2}



MODEL BERHASIL DISIMPAN


In [98]:
# =========================================================
# TEST GENERATE
# =========================================================

text = "mereka telah berhasil keluar"

input_text = (
    "generate Siapa: "
    + text
)

inputs = tokenizer(
    input_text,
    return_tensors="pt",
    truncation=True,
    max_length=128
)

device = model.device

inputs = {
    k: v.to(device)
    for k, v in inputs.items()
}

# =========================================================
# GENERATE
# =========================================================

output_ids = model.generate(

    **inputs,

    max_new_tokens=20,

    num_beams=5,

    no_repeat_ngram_size=3,

    repetition_penalty=3.0,

    length_penalty=1.2,

    early_stopping=True
)

# =========================================================
# DECODE
# =========================================================

hasil = tokenizer.decode(
    output_ids[0],
    skip_special_tokens=True,
    clean_up_tokenization_spaces=True
)

In [99]:
# =========================================================
# OUTPUT
# =========================================================

print("\n" + "=" * 50)
print("HASIL GENERATE")
print("=" * 50)

print("INPUT :")
print(text)

print("\nOUTPUT :")
print(hasil)


HASIL GENERATE
INPUT :
mereka telah berhasil keluar

OUTPUT :
Siapa yang berhasil keluar?


# TESTING MODEL

In [65]:
# =========================================================
# LOAD MODEL
# =========================================================

model_path = "model_indobart_qg"

tokenizer = AutoTokenizer.from_pretrained(
    model_path
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_path
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model.to(device)

print("Model loaded!")

Model loaded!


In [93]:
# =========================================================
# UPLOAD FILE TXT
# =========================================================

from google.colab import files

uploaded = files.upload()

file_name = list(uploaded.keys())[0]

Saving cerita_102.txt to cerita_102.txt


In [110]:
# =========================================================
# BACA FILE
# =========================================================

with open(file_name, "r", encoding="utf-8") as f:
    text = f.read()

print(text[:1000])

Pada zaman dahulu kala, tinggallah sebuah keluarga. Ayah, Ibu, serta tiga anak mereka. Suatu ketiga, sang Ayah meninggal dunia. Ia dimakamkan di pemakaman umum tepi kota yang cukup jauh. Pada hari ketujuh setelah sang Ayah wafat, Ibu ketiga anak itu masih merasa sedih kehilangan suaminya. Ia ingin membawa bunga ke makam suaminya. Maka ia berpesan pada ketiga anaknya. “Ibu akan pergi membawa bunga untuk di makam Ayah. Selama Ibu pergi, kalian harus saling menjaga, ya. Sebab ada raksasa tua yang jahat. Dia pandai menyamar menjadi manusia. Dia bisa meniru suara dan rupa. Kalian harus berhati-hati. Siapapun yang datang, jangan bukakan pintu.” Ketiga anak itu berjanji akan berhati-hati. Mereka mengantar ibu mereka sampai di muka rumah. “Adik-adik, ayo, cepat masuk rumah lagi,” kata si anak pertama pada kedua adiknya. Sang Ibu memerhatikan ketiga anaknya dari jauh dengan agak cemas. Namun ia ingin sekali membawa bunga untuk makam suaminya. Maka, ia pun pergi dengan hati gelisah. Baru saja sa

In [111]:
# =========================================================
# SPLIT KALIMAT
# =========================================================

kalimat_list = re.split(r'(?<=[.!?])\s+', text)

kalimat_list = [
    k.strip()
    for k in kalimat_list
    if len(k.strip()) > 3
]

print("\nJumlah kalimat:", len(kalimat_list))


Jumlah kalimat: 137


In [112]:
# =========================================================
# FUNCTION GENERATE
# =========================================================

def generate_question(text, tipe):

    input_text = f"generate {tipe}: {text}"

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )

    inputs = {
        k: v.to(device)
        for k, v in inputs.items()
    }

    output_ids = model.generate(

        **inputs,

        max_new_tokens=32,

        num_beams=5,

        no_repeat_ngram_size=3,

        repetition_penalty=3.0,

        length_penalty=1.2,

        early_stopping=True
    )

    hasil = tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True
    )

    return hasil

In [113]:
def detect_relevant_types(text):

    text = text.lower()

    tipe = []

    # selalu bisa
    tipe.append("siapa")
    tipe.append("apa")

    # lokasi
    if " di " in f" {text} ":
        tipe.append("dimana")

    # tujuan
    if " ke " in f" {text} ":
        tipe.append("kemana")

    # asal
    if " dari " in f" {text} ":
        tipe.append("darimana")

    # waktu
    waktu_keywords = [
        "pagi",
        "siang",
        "malam",
        "sore",
        "kemarin",
        "besok",
        "hari ini",
        "minggu lalu",
        "tahun lalu",
        "bulan lalu",
        "hari ini",
        "senin",
        "selasa",
        "rabu",
        "kamis",
        "jumat",
        "sabtu",
        "minggu",
        "jam",
        "menit",
        "detik",
        "hari",
        "minggu",
        "tahun",
        "bulan",
        "detik",
        "menit",
        "pukul"
    ]

    if any(k in text for k in waktu_keywords):
        tipe.append("kapan")

    return tipe

In [114]:
# =========================================================
# TIPE PERTANYAAN
# =========================================================

tipe_list = [
    "siapa",
    "apa",
    "dimana",
    "kemana",
    "darimana",
    "kapan"
]

In [115]:
# =========================================================
# GENERATE SEMUA
# =========================================================

hasil = []

for kalimat in kalimat_list:

    print("\n")
    print("=" * 60)
    print("KALIMAT:")
    print(kalimat)

    relevant_types = detect_relevant_types(
    kalimat
    )

    for tipe in relevant_types:

        try:

            pertanyaan = generate_question(
                kalimat,
                tipe
            )

            hasil.append({

                "kalimat": kalimat,

                "tipe": tipe,

                "pertanyaan": pertanyaan
            })

            print(f"\n[{tipe.upper()}]")
            print(pertanyaan)

        except Exception as e:

            print(f"ERROR {tipe}: {e}")



KALIMAT:
Pada zaman dahulu kala, tinggallah sebuah keluarga.

[SIAPA]
Siapa yang tinggallah sebuah keluarga dahulu kala,?

[APA]
Apa yang sebuah keluarga dahulu kala,?


KALIMAT:
Ayah, Ibu, serta tiga anak mereka.

[SIAPA]
Siapa yang tiga anak mereka?

[APA]
Apa yang tiga anak mereka?


KALIMAT:
Suatu ketiga, sang Ayah meninggal dunia.

[SIAPA]
Siapa yang meninggal dunia?

[APA]
Apa yang meninggal dunia?


KALIMAT:
Ia dimakamkan di pemakaman umum tepi kota yang cukup jauh.

[SIAPA]
Siapa yang dimakamkan di pemakaman umum tepi kota yang cukup?

[APA]
Apa yang tepi kota yang cukup?

[DIMANA]
Di mana Ia dimakamkan?


KALIMAT:
Pada hari ketujuh setelah sang Ayah wafat, Ibu ketiga anak itu masih merasa sedih kehilangan suaminya.

[SIAPA]
Siapa yang merasa pada hari?

[APA]
Apa yang Ibu ketujuh rasa pada hari?

[KAPAN]
Kapan  Ibu ketujuh?


KALIMAT:
Ia ingin membawa bunga ke makam suaminya.

[SIAPA]
Siapa yang ingin membawa bunga ke makam suam nya?

[APA]
Apa yang Ia bawa ke makam suam nya

In [109]:
# =========================================================
# DATAFRAME
# =========================================================

df_hasil = pd.DataFrame(hasil)

print("\n")
print(df_hasil.head(12))

ValueError: DataFrame constructor not properly called!

In [ ]:
# =========================================================
# SIMPAN CSV
# =========================================================

df_hasil.to_csv(
    "machine learning.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nCSV berhasil disimpan!")

#  EVALUASI BLEU SCORE

In [100]:
!pip install nltk evaluate -q

In [101]:
import pandas as pd

from nltk.translate.bleu_score import (
    sentence_bleu,
    SmoothingFunction
)

In [103]:
test_df = pd.read_csv("H6_test_dataset.csv")

In [104]:
hasil_bleu = []

for i in range(len(test_df)):

    input_text = test_df.iloc[i]["input"]

    reference = test_df.iloc[i]["target"]

    # tokenize input
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )

    # pindah ke GPU
    inputs = {
        k: v.to(model.device)
        for k, v in inputs.items()
    }

    # generate
    output_ids = model.generate(

        **inputs,

        max_new_tokens=20,

        num_beams=5,

        no_repeat_ngram_size=3,

        repetition_penalty=3.0,

        early_stopping=True
    )

    # decode
    prediction = tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )

    hasil_bleu.append({
        "input": input_text,
        "reference": reference,
        "prediction": prediction
    })

print("Selesai generate")

Selesai generate


In [105]:
df_bleu = pd.DataFrame(hasil_bleu)

df_bleu.head()

,input,reference,prediction
0,generate apa: Doni mengantarkan singkong rebus...,Apa yang Doni antarkan ke pos ronda?,Apa yang Doni antarkan ke pos ronda?
1,generate kemana: Mereka membawa barang belanja...,Ke mana Mereka membawa barang belanjaan?,Ke mana Mereka membawa barang belanjaan?
2,generate siapa: Andi mengucapkan terima kasih ...,Siapa yang mengucapkan terima kasih kepada ayah?,Siapa yang Andi ucapkan kepada ayah?
3,generate dimana: Adik bermain bola di halaman ...,Di mana Adik bermain bola?,Di mana Adik bermain bola?
4,generate siapa: Banyak penonton duduk di kursi...,Siapa yang duduk di kursi melingkar?,Siapa yang duduk di kursi melingkar?


In [106]:
smooth = SmoothingFunction().method1

bleu1_scores = []
bleu2_scores = []
bleu4_scores = []

for i in range(len(df_bleu)):

    reference = [
        df_bleu.iloc[i]["reference"].split()
    ]

    candidate = (
        df_bleu.iloc[i]["prediction"].split()
    )

    # BLEU-1
    bleu1 = sentence_bleu(
        reference,
        candidate,
        weights=(1, 0, 0, 0),
        smoothing_function=smooth
    )

    # BLEU-2
    bleu2 = sentence_bleu(
        reference,
        candidate,
        weights=(0.5, 0.5, 0, 0),
        smoothing_function=smooth
    )

    # BLEU-4
    bleu4 = sentence_bleu(
        reference,
        candidate,
        weights=(0.25, 0.25, 0.25, 0.25),
        smoothing_function=smooth
    )

    bleu1_scores.append(bleu1)
    bleu2_scores.append(bleu2)
    bleu4_scores.append(bleu4)

In [107]:
avg_bleu1 = sum(bleu1_scores) / len(bleu1_scores)
avg_bleu2 = sum(bleu2_scores) / len(bleu2_scores)
avg_bleu4 = sum(bleu4_scores) / len(bleu4_scores)

print("=" * 50)
print("HASIL BLEU SCORE")
print("=" * 50)

print(f"BLEU-1 : {avg_bleu1:.4f}")
print(f"BLEU-2 : {avg_bleu2:.4f}")
print(f"BLEU-4 : {avg_bleu4:.4f}")

HASIL BLEU SCORE
BLEU-1 : 0.7610
BLEU-2 : 0.7241
BLEU-4 : 0.6127


In [108]:
df_bleu["BLEU-1"] = bleu1_scores
df_bleu["BLEU-2"] = bleu2_scores
df_bleu["BLEU-4"] = bleu4_scores

df_bleu.to_csv(
    "7hasil_bleu_ML.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Hasil BLEU disimpan")

Hasil BLEU disimpan
